# 📝 과제 02: 온라인 쇼핑 상담 서버 가속하기

## 만들 결과

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">교안 02(추론 가속)의 sweep 측정을 과제 01의 온라인 쇼핑 상담 서버에 적용합니다. 서버 설정 두 개(기준, MTP)를 RunPod에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">vllm bench sweep serve</code> 명령 하나로 측정합니다. 끝으로 같은 파드의 노트북에서 결과를 읽어 예상 동시 요청 수에 맞는 설정을 고릅니다.</p>
<img style="display:block;max-width:100%;height:auto;margin:0;" src="images/30_gwaje02_pipeline.png" width="1050" alt="과제 02의 세 단계: 설정 파일과 조건 파일, RunPod에서 sweep 실행, 서버 설정 정하기">

<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">만들 것</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">서버 설정 딕셔너리 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serve_params</code>, 벤치마크 조건 리스트 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">bench_params</code>와 두 JSON 파일</li>
<li style="margin:3px 0;">RunPod sweep 측정 네 번(설정 2개 x 조건 2개)의 비교 행 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">sweep_rows</code>와 비교 테이블 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">sweep_table</code></li>
<li style="margin:3px 0;">고른 서버 설정 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">best_setting</code>과 결정 근거 테이블 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">decision_table</code></li>
</ul>
</div>

## 준비: 데이터와 제공 코드

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">환경:</strong> VS Code Remote SSH로 접속한 RunPod의 원격 창에서 노트북과 터미널을 실행합니다. 내 PC에서 실행하는 코드는 없습니다. 원격 저장소의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.venv</code> 커널을 고르고, 교안 01에서 키를 생성해 저장한 원격 저장소의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>를 그대로 씁니다. 파드를 새로 만들어 키가 없다면 교안 01의 준비부터 마칩니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">자료:</strong> 과제 01에서 같은 파드의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output/commerce_prompts.jsonl</code>로 저장한 벤치마크 입력 파일입니다. 그래서 과제 01을 먼저 마쳐야 합니다. 새 파드에서 시작했다면 과제 01의 입력 파일 생성 단계까지 다시 실행합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">제공 코드:</strong> 아래 첫 셀은 교안 02 1-1과 같은 라이브러리, 경로(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output_dir</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">sweep_dir</code>), 설정값(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">TTFT_TARGET_MS</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">TPOT_TARGET_MS</code> 등)입니다. 둘째 셀은 교안 02 7-2와 같은 함수 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">bench_row(setting, concurrency, result)</code>와, 3번에서 결정 근거 테이블을 보여 줄 열 목록 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">compare_columns</code>입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">교안 02와 다른 점:</strong> 입력 파일은 온라인 쇼핑 상담 것입니다. 설정 파일 이름은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">commerce_</code>로 시작하고, 실험 이름은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">commerce-settings</code>입니다. 서버 설정은 기준과 MTP 두 개, 벤치마크 조건은 동시 요청 1과 64 두 개입니다. 서버 옵션, 벤치마크 옵션, 지연 목표(TTFT 1000ms, TPOT 50ms)는 교안 02와 같습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">작성:</strong> 코드 셀은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;"># TODO</code> 주석 아래의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">...</code>를 채우고 바로 아래 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">[자가채점]</code> 셀을 실행합니다. RunPod 터미널 명령은 블록의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&lt;...&gt;</code> 칸을 요구사항의 값으로 바꿔 실행합니다. 뒤 문항이 앞 문항의 결과를 쓰므로 1번부터 차례로 풉니다.</li>
</ul>
</div>

**셀 실행 · RunPod 원격 노트북** (VS Code 원격 창의 노트북. 내 PC가 아니라 RunPod에서 실행됩니다)

In [ ]:
# ---- 라이브러리, 경로, 설정값 ----
import json
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from transformers import AutoTokenizer

material_dir = Path(".").resolve()  # 이 단원 폴더(정답 노트북은 "..")
data_dir = material_dir / "data"
output_dir = material_dir / "output"
bench_dir = output_dir / "bench"           # 같은 파드에서 벤치마크가 저장한 결과 JSON
sweep_dir = output_dir / "sweep"           # 같은 파드에서 sweep이 저장한 summary.csv
bench_dir.mkdir(parents=True, exist_ok=True)
sweep_dir.mkdir(parents=True, exist_ok=True)
# 터미널에서 생성해 원격 저장소의 .env에 저장한 키를 읽습니다
load_dotenv(material_dir.parent / ".env", override=True)
SERVER_URL = "http://127.0.0.1:8000"  # 노트북과 서버가 같은 RunPod에 있습니다
VLLM_API_KEY = os.environ["VLLM_API_KEY"]

# 설정값
MODEL_NAME = "Qwen/Qwen3.5-4B"            # 서버가 불러온 허브 모델(토크나이저도 이 이름으로 받습니다)
SERVED_NAME = "qwen3.5-4b"               # 서버 이름(--served-model-name). 요청의 model에 넣습니다
MAX_MODEL_LEN = 8192                  # 참고 조문이 든 시스템 메시지 + 질문 + 답을 담는 최대 길이
MAX_TOKENS = 256                     # 요청마다 생성 상한(지난 단원과 같은 값)
TTFT_TARGET_MS = 1000                # 지연 목표: 첫 토큰까지 1000ms 안
TPOT_TARGET_MS = 50                  # 지연 목표: 출력 토큰 하나에 50ms 안

# 테이블의 긴 답변이 잘리지 않게 칸의 글자 수 제한을 풉니다
pd.set_option("display.max_colwidth", None)

# 그래프 글꼴: 영문, 숫자, 기호는 DejaVu Sans, 한글은 운영체제의 한글 글꼴이 맡습니다
import platform

import matplotlib.pyplot as plt
from matplotlib import font_manager

if platform.system() == "Windows":
    korean_font = "Malgun Gothic"
elif platform.system() == "Darwin":
    korean_font = "AppleGothic"
else:
    korean_font = "NanumGothic"
    # 리눅스: 글꼴을 설치하기 전에 만든 matplotlib 글꼴 목록에는 나눔고딕이 없으므로, 설치된 파일을 직접 등록합니다
    for font_path in font_manager.findSystemFonts():
        if "NanumGothic" in font_path:
            font_manager.fontManager.addfont(font_path)
plt.rcParams["font.family"] = ["DejaVu Sans", korean_font]
plt.rcParams["axes.unicode_minus"] = False  # 음수 부호가 네모로 깨지지 않게 합니다

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">아래 셀의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">bench_row</code>는 교안 02 7-2와 같은 함수로, 결과 JSON 딕셔너리나 summary.csv의 한 행에서 비교 테이블의 한 행을 만듭니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">compare_columns</code>는 3번에서 결정 근거 테이블을 보여 줄 때 고르는 열입니다.</p>
</div>

In [ ]:
def bench_row(setting, concurrency, result):
    """결과 JSON 딕셔너리나 summary.csv의 한 행에서 비교 테이블의 한 행을 만듭니다."""
    return {
        "설정": setting,
        "동시 요청 수": concurrency,
        "출력 처리량(토큰/초)": result["output_throughput"],
        "p95 TTFT(ms)": result["p95_ttft_ms"],
        "p95 TPOT(ms)": result["p95_tpot_ms"],
        "goodput(요청/초)": result["request_goodput"],
        "평균 TTFT(ms)": result["mean_ttft_ms"],
        "평균 TPOT(ms)": result["mean_tpot_ms"],
        "목표 달성률": result["request_goodput"] / result["request_throughput"],  # 두 지연 목표를 모두 지킨 요청의 비율
    }


# 3번 결정 근거 테이블에서 보여 줄 열
compare_columns = ["설정", "동시 요청 수", "출력 처리량(토큰/초)", "p95 TTFT(ms)", "p95 TPOT(ms)", "goodput(요청/초)"]

## 1. 서버 설정 파일과 벤치마크 조건 파일

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적:</strong> 설정마다 서버를 다시 띄워 같은 부하로 측정하는 반복을 sweep 명령 하나에 맡기려고, 비교할 서버 설정과 벤치마크 조건을 같은 파드의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output/</code> 폴더에 JSON 파일로 저장합니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">요구사항</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">(1) <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">serve_params</code></strong>(dict): 키는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;기준&quot;</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;MTP&quot;</code> 순서입니다. 값은 그 설정이 서버 명령 뒤에 더할 옵션을 교안 02 3-1의 규칙대로 쓴 파이썬 딕셔너리입니다. 키는 옵션 이름의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">-</code>를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">_</code>로 바꾼 것이고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">speculative_config</code>의 값은 JSON 문자열이 아니라 딕셔너리로 씁니다(문자열로 바꾸는 일은 sweep이 합니다). 기준은 옵션 없음(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">{}</code>), MTP는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--speculative-config '{&quot;method&quot;: &quot;mtp&quot;, &quot;num_speculative_tokens&quot;: 2}'</code>입니다.</li>
<li style="margin:3px 0;">(2) <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">bench_params</code></strong>(list): 키가 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;dataset_path&quot;</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;max_concurrency&quot;</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;num_prompts&quot;</code>인 딕셔너리 두 개입니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;dataset_path&quot;</code>는 모두 RunPod 단원 폴더 기준 입력 파일 경로 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;output/commerce_prompts.jsonl&quot;</code>입니다. (동시 요청 수, 요청 수)는 차례로 (1, 8), (64, 64)입니다. 요청 수는 과제 01 3번과 같습니다.</li>
<li style="margin:3px 0;">(3) <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">serve_params_path</code></strong>: <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output_dir / &quot;commerce_serve_params.json&quot;</code>입니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serve_params</code>를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">json.dumps(..., ensure_ascii=False, indent=2)</code>로 바꿔 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">write_text(..., encoding=&quot;utf-8&quot;)</code>로 이 파일에 저장합니다(설정 이름 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;기준&quot;</code>이 한글이라 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ensure_ascii=False</code>로 한글을 그대로 저장합니다).</li>
<li style="margin:3px 0;">(4) <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">bench_params_path</code></strong>: <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output_dir / &quot;commerce_bench_params.json&quot;</code>입니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">bench_params</code>를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">json.dumps(..., indent=2)</code>로 바꿔 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">write_text(..., encoding=&quot;utf-8&quot;)</code>로 이 파일에 저장합니다.</li>
</ul>
</div>

<details><summary>힌트</summary>

```text
접근방법: 교안 02의 3-2를 참고합니다.
서버 설정은 이름을 키로, 더할 옵션의 딕셔너리를 값으로 둡니다. 옵션 이름의 -는 _로 바꿉니다.
벤치마크 조건은 이름 없이 딕셔너리를 리스트에 담고, 조건마다 입력 파일 경로, 동시 요청 수, 요청 수를 둡니다.
json.dumps로 만든 문자열을 Path의 write_text(..., encoding="utf-8")로 저장합니다.
```

</details>

In [ ]:
# TODO (1): 비교할 서버 설정을 이름별로, 서버 명령 뒤에 더할 옵션의 딕셔너리로 쓰세요
serve_params = {
    "기준": ...,
    "MTP": ...,
}

# TODO (2): 서버마다 보낼 벤치마크 조건(입력 파일, 동시 요청 수, 보낼 요청 수)을 쓰세요
bench_params = [
    ...,
    ...,
]

# TODO (3): 서버 설정 파일 경로를 정하고 serve_params를 JSON으로 저장하세요
serve_params_path = ...
...

# TODO (4): 벤치마크 조건 파일 경로를 정하고 bench_params를 JSON으로 저장하세요
bench_params_path = ...
...
print("서버 설정:", list(serve_params), "| 벤치마크 조건:", len(bench_params), "개")

In [ ]:
# [자가채점] 서버 설정의 이름, 순서, 옵션과 벤치마크 조건, 두 JSON 파일을 확인합니다
assert isinstance(serve_params, dict), "serve_params는 설정 이름을 키로 한 딕셔너리입니다."
assert list(serve_params) == ["기준", "MTP"], f"serve_params의 키가 {list(serve_params)}입니다. \"기준\", \"MTP\" 순서로 두세요."
assert serve_params["기준"] == {}, f"기준 설정이 {serve_params['기준']}입니다. 옵션을 바꾸지 않으므로 빈 딕셔너리입니다."
assert serve_params["MTP"] == {"speculative_config": {"method": "mtp", "num_speculative_tokens": 2}}, f"MTP 설정이 {serve_params['MTP']}입니다. 키 speculative_config에 딕셔너리 {{\"method\": \"mtp\", \"num_speculative_tokens\": 2}}를 넣으세요."

# 벤치마크 조건: 같은 입력 파일, (동시 요청 1, 요청 8), (동시 요청 64, 요청 64)
expected_bench_params = [
    {"dataset_path": "output/commerce_prompts.jsonl", "max_concurrency": 1, "num_prompts": 8},
    {"dataset_path": "output/commerce_prompts.jsonl", "max_concurrency": 64, "num_prompts": 64},
]
assert isinstance(bench_params, list) and len(bench_params) == 2, "bench_params는 조건 딕셔너리 두 개를 담은 리스트입니다(이름을 붙이지 않습니다)."
assert bench_params == expected_bench_params, f"bench_params가 {bench_params}입니다. 키 dataset_path는 \"output/commerce_prompts.jsonl\", max_concurrency와 num_prompts는 (1, 8), (64, 64) 순서로 두세요."

# 두 JSON 파일: 경로와 저장한 내용
assert serve_params_path == output_dir / "commerce_serve_params.json", f"serve_params_path가 {serve_params_path}입니다. output_dir / \"commerce_serve_params.json\"으로 두세요."
assert bench_params_path == output_dir / "commerce_bench_params.json", f"bench_params_path가 {bench_params_path}입니다. output_dir / \"commerce_bench_params.json\"으로 두세요."
assert serve_params_path.exists() and json.loads(serve_params_path.read_text(encoding="utf-8")) == serve_params, "commerce_serve_params.json의 내용이 serve_params와 다릅니다. json.dumps(serve_params, ...)를 파일에 저장했는지 확인하세요."
assert bench_params_path.exists() and json.loads(bench_params_path.read_text(encoding="utf-8")) == bench_params, "commerce_bench_params.json의 내용이 bench_params와 다릅니다. json.dumps(bench_params, ...)를 파일에 저장했는지 확인하세요."
print("✅ 통과!")

## 2. RunPod에서 sweep 실행과 summary.csv 읽기

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적:</strong> 측정 네 번(설정 2개 x 조건 2개)을 명령 하나로 돌리려고, RunPod의 원격 터미널에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">vllm bench sweep serve</code>를 실행하고 같은 파드의 노트북에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">summary.csv</code>를 비교 테이블로 만듭니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">요구사항</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">(1) <strong style="font-size:1.13em;color:#1f3a5f;">RunPod 터미널 1의 sweep 명령</strong>: 아래 명령의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&lt;...&gt;</code> 네 곳을 채워 실행합니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&lt;서버 설정 파일&gt;</code>에는 1번에서 저장한 서버 설정 파일 이름을 씁니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&lt;벤치마크 조건 파일&gt;</code>에는 벤치마크 조건 파일 이름을 씁니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&lt;실험 이름&gt;</code>은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--experiment-name</code>과 로그 파일 이름 두 곳에 같은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">commerce-settings</code>를 씁니다. 나머지 옵션은 교안 02 3-3과 같습니다.</li>
<li style="margin:3px 0;">(2) <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">sweep_results</code></strong>(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">pd.DataFrame</code>): 같은 파드에 저장된 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">sweep_dir / &quot;commerce-settings&quot; / &quot;summary.csv&quot;</code>를 읽은 테이블입니다. 첫 열은 이름 없는 행 번호이므로 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">pd.read_csv(..., index_col=0)</code>으로 읽습니다. 측정 하나가 한 행이라 4행(설정 2개 x 조건 2개)입니다.</li>
<li style="margin:3px 0;">(3) <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">sweep_rows</code></strong>(list): <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">sweep_results.to_dict(&quot;records&quot;)</code>의 행마다 교안 02 7-2와 같은 방식으로 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">bench_row</code>를 불러 딕셔너리를 만듭니다. 그 딕셔너리에 키 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;수용 길이&quot;</code>(값은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">result[&quot;spec_decode_acceptance_length&quot;]</code>)를 더합니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">summary.csv</code> 행 순서대로 4개를 담고, 값은 반올림하지 않습니다.</li>
<li style="margin:3px 0;">(4) <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">sweep_table</code></strong>(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">pd.DataFrame</code>): <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">sweep_rows</code>로 만들고 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.round(2)</code>로 반올림한 테이블입니다. 열은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;설정&quot;</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;동시 요청 수&quot;</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;출력 처리량(토큰/초)&quot;</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;p95 TTFT(ms)&quot;</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;p95 TPOT(ms)&quot;</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;goodput(요청/초)&quot;</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;평균 TTFT(ms)&quot;</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;평균 TPOT(ms)&quot;</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;목표 달성률&quot;</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;수용 길이&quot;</code> 순서입니다. 수용 길이는 MTP 행에만 값이 있고 기준 행은 빈칸(NaN)입니다.</li>
</ul>
</div>

<details><summary>힌트</summary>

```text
접근방법: 교안 02의 3-3을 참고합니다.
서버 설정 파일과 벤치마크 조건 파일은 1번에서 같은 파드의 output 폴더에 저장한 두 파일의 이름입니다.
실험 이름은 결과 폴더 이름과 로그 파일 이름이 되므로 두 칸에 같은 이름을 씁니다.
```

</details>

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">상담 서버가 아직 떠 있으면 RunPod 터미널 1에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Ctrl+C</code>로 끄고 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">nvidia-smi</code>로 GPU 메모리가 반환됐는지 확인합니다. sweep이 같은 GPU와 포트 8000에 측정용 서버를 차례로 띄우기 때문입니다. 교안 02의 sweep을 실행한 터미널 1을 그대로 쓰면 이미 단원 폴더 안이므로 아래 준비 블록은 건너뜁니다.</p>
</div>

**복사해서 실행 · RunPod 터미널 1**

```bash
# VS Code에서 연 원격 저장소(실습자료) 폴더의 터미널에서 실행합니다
cd day76_추론가속_서빙벤치마킹
pwd   # 끝이 day76_추론가속_서빙벤치마킹이면 맞습니다
mkdir -p output/bench output/sweep
# 같은 원격 저장소의 .env에서 이미 생성한 키를 불러옵니다
VLLM_API_KEY="$(uv run python -c 'from dotenv import dotenv_values; print(dotenv_values("../.env")["VLLM_API_KEY"])')" &&
export VLLM_API_KEY OPENAI_API_KEY="$VLLM_API_KEY"
```

**복사해서 실행 · RunPod 터미널 1**

```bash
uv run vllm bench sweep serve \
  --serve-cmd 'vllm serve Qwen/Qwen3.5-4B --served-model-name qwen3.5-4b --max-model-len 8192 --language-model-only --reasoning-parser qwen3 --default-chat-template-kwargs "{\"enable_thinking\": false}" --enable-prompt-tokens-details --enable-prefix-caching --host 127.0.0.1 --port 8000' \
  --bench-cmd 'vllm bench serve --backend openai --base-url http://127.0.0.1:8000 --model qwen3.5-4b --tokenizer Qwen/Qwen3.5-4B --dataset-name custom --dataset-path output/commerce_prompts.jsonl --skip-chat-template --custom-output-len 256 --temperature 0 --seed 0 --metric-percentiles 50,95,99 --goodput ttft:1000 tpot:50' \
  --serve-params output/<서버 설정 파일> \
  --bench-params output/<벤치마크 조건 파일> \
  --num-runs 1 \
  --server-ready-timeout 600 \
  --output-dir output/sweep \
  --experiment-name <실험 이름> \
  --show-stdout 2>&1 | tee output/sweep/<실험 이름>.log
```

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">sweep은 uv run 아래에서 실행되므로 안쪽 명령은 vllm으로 바로 시작합니다. 두 설정의 서버를 차례로 띄우고 끄므로 시간이 걸립니다.</p>
<p style="margin:6px 0;">같은 실험 이름 폴더가 있으면 sweep이 시작하자마자 멈춥니다. 중단된 실행을 이어 하려면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--show-stdout</code> 앞에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--resume</code>을 넣고, 끝의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">tee</code>를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">tee -a</code>로 바꿉니다. 처음부터 다시 하려면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">mv output/sweep/commerce-settings output/sweep/commerce-settings-old</code>로 앞 실험 폴더를 옮겨 둔 뒤 같은 명령을 실행합니다.</p>
<p style="margin:6px 0;">sweep이 끝나 터미널 1에 다시 명령을 입력할 수 있게 되면, 같은 원격 창의 노트북에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output/sweep/commerce-settings/summary.csv</code>를 바로 읽습니다.</p>
</div>

<details><summary>힌트</summary>

```text
접근방법: 교안 02의 7-2를 참고합니다.
summary.csv의 첫 열은 행 번호이므로 index_col=0으로 읽습니다.
행마다 bench_row에 설정 이름 열, 동시 요청 수 열, 행 전체를 넘기고 수용 길이 열을 더합니다.
```

</details>

**셀 실행 · RunPod 원격 노트북** (VS Code 원격 창의 노트북. 내 PC가 아니라 RunPod에서 실행됩니다)

In [ ]:
# TODO (2): 같은 파드의 summary.csv를 읽으세요(첫 열은 행 번호, 측정 하나가 한 행입니다)
sweep_results = ...

# TODO (3): 행마다 bench_row로 비교 테이블의 한 행을 만들고 수용 길이를 더하세요
sweep_rows = []
for result in sweep_results.to_dict("records"):
    row = ...
    row["수용 길이"] = ...
    sweep_rows.append(row)

# TODO (4): 소수 둘째 자리로 반올림한 비교 테이블을 만드세요
sweep_table = ...
display(sweep_table)

In [ ]:
# [자가채점] summary.csv 행 수, 설정 이름과 동시 요청 수, sweep_rows와 summary.csv 값의 일치, 테이블 모양, 수용 길이를 확인합니다
assert isinstance(sweep_results, pd.DataFrame), "sweep_results는 summary.csv를 pd.read_csv로 읽은 테이블입니다."
assert "Unnamed: 0" not in sweep_results.columns, "sweep_results에 Unnamed: 0 열이 있습니다. 첫 열은 행 번호이므로 pd.read_csv(..., index_col=0)으로 읽으세요."
assert len(sweep_results) == len(serve_params) * len(bench_params) == 4, f"summary.csv가 {len(sweep_results)}행입니다. 설정 2개 x 조건 2개 = 4행이어야 합니다. 원격 터미널의 sweep이 끝까지 돌았는지, 읽은 파일이 output/sweep/commerce-settings 폴더의 것인지 확인하세요."
assert set(sweep_results["_benchmark_name"]) == set(serve_params), f"summary.csv의 설정 이름이 {sorted(set(sweep_results['_benchmark_name']))}입니다. sweep 명령의 --serve-params에 1번의 서버 설정 파일을 넣었는지 확인하세요."
assert set(sweep_results["max_concurrency"]) == {1, 64}, f"summary.csv의 동시 요청 수가 {sorted(set(sweep_results['max_concurrency']))}입니다. sweep 명령의 --bench-params에 1번의 벤치마크 조건 파일을 넣었는지 확인하세요."
assert set(sweep_results["dataset_path"]) == {"output/commerce_prompts.jsonl"}, f"summary.csv의 입력 파일이 {sorted(set(sweep_results['dataset_path']))}입니다. 읽은 경로가 원격 단원 폴더의 output/sweep/commerce-settings/summary.csv인지 확인하세요."

# sweep_rows: summary.csv 행 순서대로, 반올림 전 값
assert isinstance(sweep_rows, list) and len(sweep_rows) == len(sweep_results), f"sweep_rows는 summary.csv 행마다 하나씩 {len(sweep_results)}개를 담은 리스트입니다."
for row, result in zip(sweep_rows, sweep_results.to_dict("records")):
    assert row["설정"] == result["_benchmark_name"] and row["동시 요청 수"] == result["max_concurrency"], f"sweep_rows의 행 {row['설정']!r}, {row['동시 요청 수']!r}가 summary.csv의 _benchmark_name, max_concurrency와 다릅니다. bench_row에 두 열 값을 넘기세요."
    assert row["출력 처리량(토큰/초)"] == result["output_throughput"], f"{row['설정']} 동시 요청 {row['동시 요청 수']} 행의 출력 처리량이 summary.csv의 output_throughput과 다릅니다. bench_row에 결과 행 result를 넘기고 반올림하지 마세요."
    assert row["목표 달성률"] == result["request_goodput"] / result["request_throughput"], f"{row['설정']} 행의 목표 달성률이 summary.csv 값으로 계산한 값과 다릅니다. bench_row를 그대로 쓰세요."
    if pd.isna(result["spec_decode_acceptance_length"]):
        assert pd.isna(row["수용 길이"]), f"{row['설정']} 행의 수용 길이가 {row['수용 길이']}입니다. summary.csv의 spec_decode_acceptance_length 값(이 행은 빈칸)을 그대로 넣으세요."
    else:
        assert row["수용 길이"] == result["spec_decode_acceptance_length"], f"{row['설정']} 행의 수용 길이가 summary.csv의 spec_decode_acceptance_length와 다릅니다."

# 측정 네 번: 두 설정 x 동시 요청 1, 64
measured_pairs = []
for row in sweep_rows:
    measured_pairs.append((row["설정"], row["동시 요청 수"]))
expected_pairs = []
for setting in serve_params:
    for condition in bench_params:
        expected_pairs.append((setting, condition["max_concurrency"]))
assert sorted(measured_pairs) == sorted(expected_pairs), f"측정의 (설정, 동시 요청 수)가 {sorted(measured_pairs)}입니다. 두 설정마다 동시 요청 1, 64가 하나씩 있어야 합니다."

# sweep_table: sweep_rows를 반올림한 테이블
sweep_columns = ["설정", "동시 요청 수", "출력 처리량(토큰/초)", "p95 TTFT(ms)", "p95 TPOT(ms)", "goodput(요청/초)", "평균 TTFT(ms)", "평균 TPOT(ms)", "목표 달성률", "수용 길이"]
assert isinstance(sweep_table, pd.DataFrame), "sweep_table은 sweep_rows로 만든 pd.DataFrame입니다. `...`를 채웠는지 확인하세요."
assert list(sweep_table.columns) == sweep_columns, f"sweep_table의 열이 {list(sweep_table.columns)}입니다. bench_row의 아홉 열 뒤에 \"수용 길이\"가 오는 순서입니다."
assert sweep_table.equals(pd.DataFrame(sweep_rows).round(2)), "sweep_table이 sweep_rows를 pd.DataFrame으로 만들고 .round(2)로 반올림한 테이블과 다릅니다."

# 수용 길이: MTP 행만 1 이상 3 이하(초안 2개를 검증), 기준은 빈칸
for setting, length in zip(sweep_table["설정"], sweep_table["수용 길이"]):
    if setting == "MTP":
        assert 1 <= length <= 3, f"MTP 행의 수용 길이가 {length}입니다. 초안 2개를 검증하므로 1 이상 3 이하입니다. 1이면 초안이 한 번도 맞지 않은 것입니다. MTP 설정의 num_speculative_tokens를 확인하세요."
    else:
        assert pd.isna(length), f"{setting} 행의 수용 길이가 {length}입니다. 투기적 디코딩을 켜지 않은 서버는 빈칸(NaN)입니다."
print("✅ 통과!")

## 3. 서버 설정 정하기

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적:</strong> 상담 서버를 띄울 옵션을 측정 근거와 함께 정하려고, 예상 동시 요청 수에서 지연 목표를 지키는 설정 가운데 출력 처리량이 가장 큰 설정을 고릅니다. 예상 동시 요청 수는 작성 셀에 주어진 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">expected_concurrency = 1</code>입니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">요구사항</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">(1) <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">best_setting</code></strong>(str): 반올림 전 값인 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">sweep_rows</code>에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;동시 요청 수&quot;</code>가 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">expected_concurrency</code>이고 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;목표 달성률&quot;</code>이 0.95 이상인 행 가운데 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;출력 처리량(토큰/초)&quot;</code>이 가장 큰 행의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;설정&quot;</code>입니다. 그 행의 출력 처리량은 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">best_throughput</code></strong>(float)에 담습니다. 그런 행이 없으면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">best_setting</code>은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">None</code>으로 둡니다.</li>
<li style="margin:3px 0;">(2) <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">decision_table</code></strong>(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">pd.DataFrame</code>): <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">sweep_table</code>에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;동시 요청 수&quot;</code>가 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">expected_concurrency</code>인 행만 고른 두 행 테이블입니다. 열은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">sweep_table</code>과 같습니다.</li>
</ul>
</div>

<details><summary>힌트</summary>

```text
접근방법: 교안 02의 8-2를 참고합니다.
반복문에서 동시 요청 수, 목표 달성률, 지금까지의 최대 출력 처리량 세 조건을 함께 비교합니다.
결정 근거 테이블은 동시 요청 수 열의 비교 결과로 행만 고릅니다.
```

</details>

In [ ]:
# 서비스의 예상 동시 요청 수(주어진 값)
expected_concurrency = 1

# TODO (1): 예상 동시 요청 수의 행 가운데 목표 달성률 0.95 이상이고 출력 처리량이 가장 큰 설정을 고르세요
best_setting = None
best_throughput = 0
for row in sweep_rows:
    if ...:
        best_setting = ...
        best_throughput = ...

# TODO (2): 결정 근거 테이블로 예상 동시 요청 수의 두 설정 행을 고르세요
decision_table = ...
display(decision_table[compare_columns + ["목표 달성률"]])

# 고른 설정이 없으면 이 동시 요청 수에서는 어떤 설정도 지연 목표를 95% 이상 지키지 못한 것입니다
if best_setting is None:
    print(f"동시 요청 {expected_concurrency}에서 지연 목표를 95% 이상 지킨 설정이 없습니다. 지연 목표를 다시 정하거나, 서버 한 대에 보내는 동시 요청 수를 줄이거나 서버를 늘립니다.")
else:
    print(f"동시 요청 {expected_concurrency}에서 고른 설정: {best_setting} | 더할 옵션: {serve_params[best_setting]} | 출력 처리량: {best_throughput:.1f} 토큰/초")

In [ ]:
# [자가채점] 예상 동시 요청 수, 고른 설정과 그 처리량, 결정 근거 테이블을 확인합니다
assert expected_concurrency == 1, f"expected_concurrency가 {expected_concurrency!r}입니다. 작성 셀에 주어진 1을 그대로 둡니다."
assert best_setting is None or best_setting in serve_params, f"best_setting이 {best_setting!r}입니다. serve_params의 설정 이름 하나를 담으세요. 조건 식에서 동시 요청 수, 목표 달성률, 출력 처리량을 모두 비교했는지 확인하세요."

# sweep_rows에서 같은 규칙, 같은 순서(앞 행부터)로 다시 고른 설정과 같은지 봅니다
rule_setting = None
rule_throughput = 0
for row in sweep_rows:
    if row["동시 요청 수"] == 1 and row["목표 달성률"] >= 0.95 and row["출력 처리량(토큰/초)"] > rule_throughput:
        rule_setting = row["설정"]
        rule_throughput = row["출력 처리량(토큰/초)"]
assert best_setting == rule_setting, f"best_setting이 {best_setting}입니다. 동시 요청 1에서 목표 달성률 0.95 이상인 설정 가운데 출력 처리량이 가장 큰 설정은 {rule_setting}입니다. 반올림 전 값인 sweep_rows로 고르세요."
assert best_throughput == rule_throughput, f"best_throughput이 {best_throughput}입니다. 고른 행의 반올림 전 출력 처리량 {rule_throughput}을 담으세요."

# 결정 근거 테이블: 동시 요청 1의 두 설정, sweep_table과 같은 열과 값
assert isinstance(decision_table, pd.DataFrame), "decision_table은 sweep_table에서 행을 고른 pd.DataFrame입니다. `...`를 채웠는지 확인하세요."
assert list(decision_table.columns) == list(sweep_table.columns), "decision_table의 열이 sweep_table과 다릅니다. 열은 고르지 말고 행만 고르세요."
assert len(decision_table) == len(serve_params) and sorted(decision_table["설정"]) == sorted(serve_params), f"decision_table의 설정이 {list(decision_table['설정'])}입니다. 두 설정이 한 행씩 있어야 합니다."
assert set(decision_table["동시 요청 수"]) == {1}, f"decision_table의 동시 요청 수가 {sorted(set(decision_table['동시 요청 수']))}입니다. 동시 요청 수가 expected_concurrency인 행만 고르세요."
for setting, throughput in zip(decision_table["설정"], decision_table["출력 처리량(토큰/초)"]):
    source_rows = sweep_table[(sweep_table["설정"] == setting) & (sweep_table["동시 요청 수"] == 1)]
    assert throughput == source_rows["출력 처리량(토큰/초)"].iloc[0], f"decision_table의 {setting} 행 출력 처리량이 sweep_table의 같은 행과 다릅니다."
print("✅ 통과!")

## 마무리

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">sweep은 서버 설정마다 측정을 마치면 그 서버를 끄므로, 마지막 MTP 서버도 sweep이 끝날 때 꺼져 있습니다. RunPod 터미널 1에서 GPU 메모리가 반환됐는지 확인합니다.</p>
</div>

**RunPod 터미널 1 · 화면에서 확인**

```bash
nvidia-smi
```

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">확인 기준</strong>: GPU 메모리 사용량이 실행 전 수준입니다. 메모리가 남아 있으면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">pgrep -af &quot;vllm serve&quot;</code>로 남은 vLLM 프로세스의 PID를 찾아 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">kill &lt;PID&gt;</code>로 끕니다.</p>
</div>